# Setup


In [ ]:

%pip install -q -U "crewai[google-genai,tools]" ddgs


# Config


In [ ]:

import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY is missing. Add it in Colab Secrets "
        "and enable notebook access."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
print("Gemini API key configured successfully.")


# Model


In [ ]:

from crewai import LLM

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    temperature=0.2,
    max_tokens=4096
)

print("Gemini LLM configured.")


# Search tool


In [ ]:

from crewai.tools import tool
from ddgs import DDGS

@tool("DuckDuckGo Search")
def duckduckgo_search(query: str) -> str:
    """Search the web and return relevant results with titles,
    snippets, and source URLs for a given query."""

    try:
        results = DDGS().text(query, max_results=5)

        if not results:
            return "No search results found for this query."

        output = []

        for index, result in enumerate(results, start=1):
            output.append(
                f"Result {index}\n"
                f"Title: {result.get('title', 'N/A')}\n"
                f"Snippet: {result.get('body', 'N/A')}\n"
                f"URL: {result.get('href', 'N/A')}"
            )

        return "\n\n".join(output)

    except Exception as e:
        return f"Search failed: {type(e).__name__}: {e}"

print("DuckDuckGo search tool created.")


# Agents


In [ ]:

from crewai import Agent
from crewai_tools import FileWriterTool

file_writer = FileWriterTool()

# Agent 1: Researcher
researcher = Agent(
    role="Research Analyst",
    goal=(
        "Investigate the assigned topic via web search to gather "
        "relevant facts, valuable insights, and source URLs."
    ),
    backstory=(
        "You are an analytical researcher who strictly separates "
        "verified evidence from speculation and never creates false sources."
    ),
    tools=[duckduckgo_search],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

# Agent 2: Writer
writer = Agent(
    role="Content Writer",
    goal=(
        "Draft an engaging blog article of around 600 words "
        "based directly on the gathered research."
    ),
    backstory=(
        "You are a skilled content creator who turns raw research "
        "into well-structured, clear, and compelling articles."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3
)

# Agent 3: Editor
editor = Agent(
    role="Content Editor",
    goal=(
        "Review the draft for flow, tone, and factual accuracy, "
        "then write out the completed post to disk."
    ),
    backstory=(
        "You are a detail-oriented copy editor. You verify facts against sources, "
        "prune unsupported assertions, and use the file writer tool "
        "to export the final Markdown piece."
    ),
    tools=[file_writer],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3
)

print("All three agents created.")


# Tasks


In [ ]:

from crewai import Task

research_task = Task(
    description=(
        "Research the topic: {topic}\n\n"
        "Use DuckDuckGo search to investigate the topic. "
        "Collect the key facts, recent developments where relevant, "
        "useful examples, and at least three relevant source URLs "
        "when available. Return organized research notes and clearly "
        "distinguish verified information from uncertainty."
    ),
    expected_output=(
        "Organized research notes containing key findings, useful "
        "examples, and source titles and URLs. Do not invent citations."
    ),
    agent=researcher
)

writing_task = Task(
    description=(
        "Write a blog post of approximately 600 words about {topic}, "
        "using the research supplied by the preceding task. Include "
        "an engaging title, introduction, logical section headings, "
        "examples where useful, and a conclusion. Preserve source URLs "
        "for factual claims and avoid unsupported statistics."
    ),
    expected_output=(
        "A complete blog post draft of approximately 600 words in "
        "Markdown, with a title, headings, conclusion, and source links."
    ),
    agent=writer
)

editing_task = Task(
    description=(
        "Edit the draft produced by the preceding task about {topic}. "
        "Improve grammar, readability, tone, organization, and factual "
        "consistency. Preserve useful source links. Do not invent facts "
        "or claim that a statement is verified without supporting evidence. "
        "Save the final edited article to blog_post.md using FileWriterTool. "
        "The file must contain the article itself, not a discussion of the task."
    ),
    expected_output=(
        "The final edited Markdown blog post, with a clear title, "
        "well-organized sections, conclusion, and relevant source links. "
        "The file blog_post.md must be written successfully."
    ),
    agent=editor,
    output_file="blog_post.md"
)

print("All three tasks created.")


# Crew


In [ ]:

from crewai import Crew, Process

blog_crew = Crew(
    agents=[
        researcher,
        writer,
        editor
    ],
    tasks=[
        research_task,
        writing_task,
        editing_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Blog Post Crew assembled.")


# Input & run


In [ ]:

# Enter a topic (e.g. Edge AI in Healthcare, Smart Grid Automation)
TOPIC = input("Enter the topic for your blog post: ").strip()

if not TOPIC:
    print("Please enter a valid topic.")
else:
    result = await blog_crew.kickoff_async(
        inputs={"topic": TOPIC}
    )

    print("\nBlog post generated successfully!")
    print("Topic:", TOPIC)

# Verify output


In [ ]:

from pathlib import Path
from IPython.display import display, Markdown

output_path = Path("blog_post.md")

if output_path.exists() and output_path.stat().st_size > 0:
    final_blog = output_path.read_text(encoding="utf-8")

    print(f"File created: {output_path.resolve()}")
    print(f"File size: {output_path.stat().st_size} bytes")
    print(f"Word count: {len(final_blog.split())}")

    display(Markdown(final_blog))
else:
    print("The output file was not created or is empty.")
    print("Inspect the crew execution logs for errors.")


# Download


In [ ]:
from google.colab import files
files.download("blog_post.md")